# RQ3

In [1]:
import pandas as pd

df = pd.read_csv("../data/dataset.csv")
print(df.shape)
df.head()

(1800, 5)


,prompt_id,domain,llm_name,llm_input,llm_output
0,P001,writing,claude,Write a formal email to a professor asking for...,**Subject:** Request for Extension – Moon Proj...
1,P002,coding,claude,Explain how exception handling works in Java a...,# Exception Handling in Java\n\nException hand...
2,P003,coding,claude,Write a Go program that reads a CSV file and p...,Here's a Go program that reads a CSV file and ...
3,P004,coding,claude,Write a JavaScript function that finds the two...,"Here's an efficient solution using a hash map,..."
4,P005,coding,claude,Write a C++ program that prints the numbers 1 ...,Here's a C++ FizzBuzz program:\n\n```cpp\n#inc...


In [2]:
import numpy as np
import torch
import random


SEED=67
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

labels = {"claude": 0, "gpt": 1, "deepseek": 2}
df["label"] = df["llm_name"].map(labels)


df.head()

mps


,prompt_id,domain,llm_name,llm_input,llm_output,label
0,P001,writing,claude,Write a formal email to a professor asking for...,**Subject:** Request for Extension – Moon Proj...,0
1,P002,coding,claude,Explain how exception handling works in Java a...,# Exception Handling in Java\n\nException hand...,0
2,P003,coding,claude,Write a Go program that reads a CSV file and p...,Here's a Go program that reads a CSV file and ...,0
3,P004,coding,claude,Write a JavaScript function that finds the two...,"Here's an efficient solution using a hash map,...",0
4,P005,coding,claude,Write a C++ program that prints the numbers 1 ...,Here's a C++ FizzBuzz program:\n\n```cpp\n#inc...,0


In [3]:
ids = np.array(df["prompt_id"].unique())
rng = np.random.default_rng(SEED)
rng.shuffle(ids)

n = len(ids)
train_ids = ids[:int(0.70 * n)]
val_ids   = ids[int(0.70 * n):int(0.85 * n)]
test_ids  = ids[int(0.85 * n):]

train_df = df[df["prompt_id"].isin(train_ids)].reset_index(drop=True)
val_df   = df[df["prompt_id"].isin(val_ids)].reset_index(drop=True)
test_df  = df[df["prompt_id"].isin(test_ids)].reset_index(drop=True)

print(len(train_df), len(val_df), len(test_df))
print(train_df.groupby(["llm_name", "domain"]).size().unstack())

1260 270 270
domain    coding  math  writing
llm_name                       
claude       136   144      140
deepseek     136   144      140
gpt          136   144      140


In [4]:
import re
from collections import Counter

def tokenize(text):
    text = text.replace("\n", " <nl> ")
    return re.findall(r"<nl>|\w+|[^\w\s]", text.lower())

def build_vocab(texts, max_words=20000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in counter.most_common(max_words):
        vocab[word] = len(vocab)
    return vocab

def encode(text, vocab, max_len=400):
    ids = [vocab.get(t, 1) for t in tokenize(text)][:max_len]
    return ids + [0] * (max_len - len(ids))

# quick check
print(tokenize("## Hello!\n**Bold** `code`"))
vocab = build_vocab(train_df["llm_output"])
print("vocab size:", len(vocab))
print(encode(train_df["llm_output"][0], vocab)[:15])



['#', '#', 'hello', '!', '<nl>', '*', '*', 'bold', '*', '*', '`', 'code', '`']
vocab size: 11622
[4, 4, 1261, 11, 4, 4, 849, 35, 714, 672, 361, 617, 2, 2, 442]


In [5]:
id_to_word = {i: w for w, i in vocab.items()}
print([id_to_word[i] for i in encode(train_df["llm_output"][0], vocab)[:15]])
print(train_df["llm_output"][0][:100])

['*', '*', 'subject', ':', '*', '*', 'request', 'for', 'extension', '–', 'moon', 'project', '<nl>', '<nl>', 'dear']
**Subject:** Request for Extension – Moon Project

Dear Professor [Last Name],

I hope this message 


In [6]:
from torch.utils.data import Dataset, DataLoader

for d in (train_df, val_df, test_df):
    d["both"] = d["llm_input"] + " [SEP] " + d["llm_output"]

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.x = torch.tensor([encode(t, vocab) for t in texts])
        self.y = torch.tensor(list(labels))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.x[i], self.y[i]

def make_loaders(train_df, val_df, test_df, text_col, batch_size=32):
    vocab = build_vocab(train_df[text_col])
    train_ds = TextDataset(train_df[text_col], train_df["label"], vocab)
    val_ds   = TextDataset(val_df[text_col],   val_df["label"],   vocab)
    test_ds  = TextDataset(test_df[text_col],  test_df["label"],  vocab)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(val_ds,   batch_size=batch_size),
            DataLoader(test_ds,  batch_size=batch_size),
            vocab)

train_loader, val_loader, test_loader, vocab = make_loaders(train_df, val_df, test_df, "llm_output")
x, y = next(iter(train_loader))
print(x.shape, y.shape, y[:10])

torch.Size([32, 400]) torch.Size([32]) tensor([1, 1, 1, 2, 0, 2, 0, 2, 0, 2])


In [7]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=100,
                 kernel_sizes=(3, 4, 5), num_classes=3, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList(
            [nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):                  # x: (32, 400) token numbers
        x = self.embedding(x)              # (32, 400, 128) each token → 128 numbers
        x = x.transpose(1, 2)              # (32, 128, 400) Conv1d wants this order
        pooled = [F.relu(conv(x)).max(dim=2).values for conv in self.convs]
        x = torch.cat(pooled, dim=1)       # (32, 300) = 100 filters × 3 sizes
        return self.fc(self.dropout(x))    # (32, 3) one score per LLM

model = TextCNN(len(vocab)).to(device)
print(model(x.to(device)).shape)
print("parameters:", sum(p.numel() for p in model.parameters()))

torch.Size([32, 3])
parameters: 1642419


In [8]:
class TextLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 num_layers=1, num_classes=3, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):                                   # x: (32, 400)
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()    # real length of each text (no padding)
        emb = self.embedding(x)                             # (32, 400, 128)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)                       # h: final hidden states
        h = torch.cat([h[-2], h[-1]], dim=1)                # (32, 256) forward + backward
        return self.fc(self.dropout(h))                     # (32, 3)

lstm = TextLSTM(len(vocab)).to(device)
print(lstm(x.to(device)).shape)
print("parameters:", sum(p.numel() for p in lstm.parameters()))

torch.Size([32, 3])
parameters: 1752579


In [13]:
import copy
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def evaluate(model, loader):
    model.eval()                       # turn off dropout
    preds, labels = [], []
    with torch.no_grad():              # no learning here, just predicting
        for x, y in loader:
            out = model(x.to(device))
            preds += out.argmax(dim=1).cpu().tolist()   # highest score = predicted LLM
            labels += y.tolist()
    return preds, labels

def train_and_eval(model, train_loader, val_loader, test_loader, epochs=10, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_val, best_state, history = -1, None, []

    for epoch in range(1, epochs + 1):
        model.train()                  # turn dropout on
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()      # clear old gradients
            loss = loss_fn(model(x), y)   # how wrong the guesses are
            loss.backward()            # work out how to adjust each weight
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # stop huge updates (helps LSTM)
            optimizer.step()           # adjust the weights
            total_loss += loss.item() * len(y)

        train_loss = total_loss / len(train_loader.dataset)
        p, l = evaluate(model, val_loader)
        val_acc = accuracy_score(l, p)
        history.append((epoch, train_loss, val_acc))
        print(f"epoch {epoch:2d}  train loss {train_loss:.3f}  val acc {val_acc:.3f}")

        if val_acc > best_val:         # remember the best epoch
            best_val, best_state = val_acc, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)  # go back to the best epoch
    p, l = evaluate(model, test_loader)
    results = {"test_acc": accuracy_score(l, p),
               "test_f1": f1_score(l, p, average="macro"),
               "confusion": confusion_matrix(l, p),
               "history": history,
               "preds": p, "labels": l}          # predictions kept for RQ3/RQ4
    print(f"best val acc {best_val:.3f} | TEST acc {results['test_acc']:.3f}  macro-F1 {results['test_f1']:.3f}")
    return results

In [14]:
names = ["claude", "gpt", "deepseek"]

def per_domain_acc(res, df_):
    d = df_.assign(pred=res["preds"], true=res["labels"])
    return d.groupby("domain").apply(lambda g: accuracy_score(g.true, g.pred))

# Reference: models trained on ALL domains (like RQ1), scored per domain on the test set
ref = {}
for name, M in [("CNN", TextCNN), ("LSTM", TextLSTM)]:
    torch.manual_seed(SEED)
    tl, vl, te, vocab_ref = make_loaders(train_df, val_df, test_df, "llm_output")
    ref[name] = per_domain_acc(train_and_eval(M(len(vocab_ref)), tl, vl, te), test_df)

# Cross-domain: train on 2 domains, test on the 3rd (never seen)
rows = []
for held_out in ["writing", "coding", "math"]:
    tr  = train_df[train_df.domain != held_out].reset_index(drop=True)
    va  = val_df[val_df.domain != held_out].reset_index(drop=True)
    te_ = df[df.domain == held_out].reset_index(drop=True)      # ALL 600 answers of that domain

    for name, M in [("CNN", TextCNN), ("LSTM", TextLSTM)]:
        print(f"\n== train on others, test on {held_out} | {name} ==")
        torch.manual_seed(SEED)
        tl, vl, tel, vocab_x = make_loaders(tr, va, te_, "llm_output")
        r = train_and_eval(M(len(vocab_x)), tl, vl, tel)
        f1s = f1_score(r["labels"], r["preds"], average=None)
        rows.append({"test domain": held_out, "model": name,
                     "seen (all-domain model)": round(ref[name][held_out], 3),
                     "unseen (cross-domain)": round(r["test_acc"], 3),
                     **{f"F1 {n}": round(f, 3) for n, f in zip(names, f1s)}})

rq3_table = pd.DataFrame(rows)
print(rq3_table.to_string())

epoch  1  train loss 1.220  val acc 0.726
epoch  2  train loss 0.805  val acc 0.874
epoch  3  train loss 0.576  val acc 0.874
epoch  4  train loss 0.435  val acc 0.881
epoch  5  train loss 0.371  val acc 0.885
epoch  6  train loss 0.339  val acc 0.911
epoch  7  train loss 0.275  val acc 0.904
epoch  8  train loss 0.225  val acc 0.907
epoch  9  train loss 0.218  val acc 0.900
epoch 10  train loss 0.195  val acc 0.889
best val acc 0.911 | TEST acc 0.870  macro-F1 0.871
epoch  1  train loss 0.865  val acc 0.704
epoch  2  train loss 0.664  val acc 0.733
epoch  3  train loss 0.575  val acc 0.733
epoch  4  train loss 0.455  val acc 0.741
epoch  5  train loss 0.362  val acc 0.733
epoch  6  train loss 0.235  val acc 0.789
epoch  7  train loss 0.143  val acc 0.778
epoch  8  train loss 0.075  val acc 0.763
epoch  9  train loss 0.038  val acc 0.759
epoch 10  train loss 0.023  val acc 0.778
best val acc 0.789 | TEST acc 0.730  macro-F1 0.730

== train on others, test on writing | CNN ==
epoch  1  